# 00 · Validate your setup

Welcome to **Model Mastery** — a 90-minute journey from *picking a model* to *optimizing an agent* with Microsoft Foundry. Along the way you'll build **TrailMate**, a grounded gear expert for Contoso Outdoors, and learn to **"hill climb"**: change one thing at a time, measure it, and keep only what the evidence supports.

Before any of that, this short notebook makes sure your environment is ready. Your setup script (`setenv.sh` on Skillable, or `provision.sh` self-guided) already created a `.env` file — here we simply **confirm everything is wired up** so the rest of the labs just work.

### Learning objectives
By the end of this notebook you'll be able to:
- **Load** the workshop `.env` and confirm the required variables are present.
- **Connect** to your Foundry project with your Azure sign-in (no API keys).
- **Verify** the model deployments the labs use are available.
- **Confirm** tracing (Application Insights) is connected for observability.

> ⏱️ **~15 minutes** · This is a checkpoint — if every cell is green, you're clear to continue.


## 1 · Load your environment

The setup script wrote a single `.env` file at `foundry/agent-builder/src/.env` that **every** lab shares. Let's find it and load it. The helper below walks up the folder tree, so it works whether your notebook runs from the repo root or from this `labs/core/` folder.


In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv


def find_env_file() -> Path:
    """Walk up from the current folder to locate agent-builder/src/.env."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        # Running from the repository root.
        candidate = base / "foundry" / "agent-builder" / "src" / ".env"
        if candidate.exists():
            return candidate
        # Running from somewhere inside agent-builder (e.g. labs/core).
        if base.name == "agent-builder" and (base / "src" / ".env").exists():
            return base / "src" / ".env"
    raise FileNotFoundError(
        "Could not find src/.env. Run scripts/setenv.sh (Skillable) or "
        "scripts/provision.sh (self-guided) first, then re-run this cell."
    )


ENV_FILE = find_env_file()
load_dotenv(ENV_FILE)
print(f"Loaded environment from: {ENV_FILE}")


Loaded environment from: /workspaces/model-mastery/foundry/agent-builder/src/.env


## 2 · Check the required variables

Think of these as the workshop's "keys to the building." The setup script writes all of them; if any are missing, the labs can't talk to Foundry. We print each one (secrets masked) so you can eyeball that they look right.


In [3]:
# The keys every lab needs. setenv.sh / provision.sh write all of these.
REQUIRED_VARS = [
    "FOUNDRY_PROJECT_ENDPOINT",               # which Foundry project to talk to
    "TRAILMATE_MODEL_DEPLOYMENT",             # default chat model deployment
    "TRAILMATE_AGENT_NAME",                   # the agent we'll build in lab 02
    "APPLICATIONINSIGHTS_CONNECTION_STRING",  # tracing / observability
    "AZURE_SUBSCRIPTION_ID",
    "AZURE_RESOURCE_GROUP",
    "AZURE_AI_ACCOUNT_NAME",
    "AZURE_AI_PROJECT_NAME",
    "AZURE_LOCATION",
]


def mask(value: str) -> str:
    """Show just enough of a value to recognize it, while hiding secrets."""
    if not value:
        return "(empty)"
    return value if len(value) <= 12 else f"{value[:6]}…{value[-4:]}"


missing = [name for name in REQUIRED_VARS if not os.environ.get(name)]
for name in REQUIRED_VARS:
    status = "✅" if os.environ.get(name) else "❌"
    print(f"{status} {name:<40} {mask(os.environ.get(name, ''))}")

if missing:
    raise SystemExit(
        f"\nMissing {len(missing)} variable(s): {', '.join(missing)}.\n"
        "Re-run the setup script, then run this cell again."
    )
print("\nAll required variables are present.")


✅ FOUNDRY_PROJECT_ENDPOINT                 https:…bw2a
✅ TRAILMATE_MODEL_DEPLOYMENT               gpt-5.4
✅ TRAILMATE_AGENT_NAME                     trailmate
✅ APPLICATIONINSIGHTS_CONNECTION_STRING    Instru…6906
✅ AZURE_SUBSCRIPTION_ID                    b90b6b…11bc
✅ AZURE_RESOURCE_GROUP                     rg-mod…0515
✅ AZURE_AI_ACCOUNT_NAME                    foundr…bw2a
✅ AZURE_AI_PROJECT_NAME                    foundr…bw2a
✅ AZURE_LOCATION                           sweden…tral

All required variables are present.


## 3 · Connect to your Foundry project

No API keys here — Foundry uses your Azure sign-in. `DefaultAzureCredential` picks up the `az login` session your setup created. We create two clients:

- **`project_client`** — the Foundry SDK entry point (agents, evaluations, data).
- **`openai_client`** — an OpenAI-compatible client for calling models and, later, your agent.


In [4]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# DefaultAzureCredential reuses the session created by `az login` during setup.
credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential=credential,
)

# The OpenAI-compatible client is how we call models and (later) the agent.
openai_client = project_client.get_openai_client()

print("Connected to Foundry project:")
print(f"  {os.environ['AZURE_AI_PROJECT_NAME']}  (region: {os.environ['AZURE_LOCATION']})")


Connected to Foundry project:
  foundry-workshop-iziws3ekpbw2a  (region: swedencentral)


## 4 · Confirm your model deployments

The core labs use four deployments (`gpt-5.4`, `gpt-5.4-mini`, `model-router`, `MAI-Image-2.5-Pro`). On Skillable these — plus two **Claude** models — are pre-provisioned for you; self-guided learners deploy the four core ones with `provision.sh` (Claude is optional). We list deployments straight from your **Foundry project endpoint** and check what's present.


In [8]:
# Deployments the core labs rely on (required).
CORE_DEPLOYMENTS = ["gpt-5.4", "gpt-5.4-mini", "model-router", "MAI-Image-2.5-Pro", "MAI-Image-2.5"]
# Claude deployments — Skillable pre-provisions these; optional for self-guided.
OPTIONAL_DEPLOYMENTS = ["claude-sonnet-4-6", "claude-haiku-4-5"]

try:
    # List deployments from the Foundry PROJECT endpoint (project_client is built
    # with FOUNDRY_PROJECT_ENDPOINT), so we see exactly what this project serves.
    available = {
        getattr(d, "name", None) or getattr(d, "deployment_name", None)
        for d in project_client.deployments.list()
    }
    available.discard(None)

    print("Deployments found in your project:")
    for name in sorted(available):
        print(f"  • {name}")

    print("\nCore deployments (required):")
    for name in CORE_DEPLOYMENTS:
        mark = "✅" if name in available else "⚠️"
        print(f"  {mark} {name}")

    print("\nClaude deployments (Skillable has these; optional self-guided):")
    for name in OPTIONAL_DEPLOYMENTS:
        mark = "✅" if name in available else "➖"
        print(f"  {mark} {name}")

    absent_core = [d for d in CORE_DEPLOYMENTS if d not in available]
    if absent_core:
        print(
            f"\nHeads up: {', '.join(absent_core)} not found. On Skillable these are "
            "pre-provisioned — double-check you used the right resource group."
        )
    else:
        print("\nAll core deployments are ready.")
except Exception as err:
    # Listing can vary by SDK version — don't block setup on it.
    print(f"Could not list deployments automatically ({err}).")
    print("You can confirm them in the Foundry portal under Build → Models.")


Deployments found in your project:
  • MAI-Image-2.5
  • claude-haiku-4-5
  • claude-sonnet-4-6
  • gpt-5.4
  • gpt-5.4-mini
  • model-router
  • text-embedding-3-large

Core deployments (required):
  ✅ gpt-5.4
  ✅ gpt-5.4-mini
  ✅ model-router
  ⚠️ MAI-Image-2.5-Pro
  ✅ MAI-Image-2.5

Claude deployments (Skillable has these; optional self-guided):
  ✅ claude-sonnet-4-6
  ✅ claude-haiku-4-5

Heads up: MAI-Image-2.5-Pro not found. On Skillable these are pre-provisioned — double-check you used the right resource group.


## 5 · Confirm tracing is wired up

**Observability** is a big theme of this workshop. When we build TrailMate, every run will emit **traces** (what happened) and **metrics** (tokens, latency, quality). That flows into **Application Insights**, connected via the string below. Here we just confirm it's present and well-formed.


In [6]:
# A valid connection string starts with "InstrumentationKey=".
conn = os.environ.get("APPLICATIONINSIGHTS_CONNECTION_STRING", "")
if conn.startswith("InstrumentationKey="):
    print("✅ Application Insights is connected.")
    print("   Traces and run metrics will appear in the Foundry portal once we")
    print("   build and test the agent in lab 02.")
else:
    print("⚠️ The Application Insights connection string looks off.")
    print("   Tracing may not work — re-run the setup script if lab 02 shows no traces.")


✅ Application Insights is connected.
   Traces and run metrics will appear in the Foundry portal once we
   build and test the agent in lab 02.


## ✅ Summary — you're ready to climb

If every cell above is green, your environment is good to go. You just confirmed:

- 🔑 The workshop **`.env`** loads and all required variables are present.
- 🔗 You can **connect** to your Foundry project with your Azure sign-in.
- 🧠 The core **model deployments** (`gpt-5.4`, `gpt-5.4-mini`, `model-router`, `MAI-Image-2.5-Pro`) are available.
- 📈 **Application Insights** is connected for tracing and metrics.

Every later notebook reuses this same `.env`, so you won't need to set anything up again.

### Next
➡️ Open **[01 · Select the right model](01-model-selection.ipynb)** — where you'll learn to right-size a model by capability, cost, latency, and quality.
